# Fine-tune Mô hình Phát hiện Biển số YOLO (`litealpr_yolov8n_eff.pt`)

Notebook này huấn luyện nâng cấp trực tiếp từ trọng số **`litealpr_yolov8n_eff.pt`** (kiến trúc siêu nhẹ `C3Ghost` — chỉ `4.3 MB`, tốc độ `~5.5 ms/ảnh` trên GPU) để khắc phục triệt để các ca khó thực tế tại trạm thu phí:
- **Ban đêm thiếu sáng & lóa đèn pha gắt** chiếu ngược vào camera.
- **Biển số bị dòng chữ vàng OSD của trạm ETC in đè che 15–35% mép trên** (như ca `36H-007.16`).
- **Xe ở xa tít sát mép trên khung hình** (biển vàng/trắng nhỏ, tối màu như ca `14H-151.06`).
- **Ảnh âm tính (Hard Negatives):** Chữ vàng OSD, đèn pha, cọc phản quang không có biển số — giúp chống bắt nhầm tuyệt đối.
- **Đa dạng góc chụp cả Ô tô (1 dòng, 2 dòng, biển đỏ/ngoại giao) & Xe máy.**

👉 **Chuẩn bị trước khi chạy:** Tải 1 file duy nhất `du_lieu/tap_finetune_yolo_detect.zip` lên thư mục gốc **Google Drive (`MyDrive`)** của bạn.

In [ ]:
# Bước 1: Kết nối Google Drive và kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
# Bước 2: Cài đặt Ultralytics và giải nén gói dữ liệu tap_finetune_yolo_detect.zip
!pip install -q -U ultralytics

DUONG_DAN_ZIP = '/content/drive/MyDrive/tap_finetune_yolo_detect.zip'
!rm -rf /content/du_lieu_yolo_detect
!mkdir -p /content/du_lieu_yolo_detect
!unzip -q -o "{DUONG_DAN_ZIP}" -d /content/du_lieu_yolo_detect

from pathlib import Path
goc = Path('/content/du_lieu_yolo_detect')
print('✅ Số ảnh Train:', len(list((goc / 'images/train').glob('*.jpg'))))
print('✅ Số ảnh Val  :', len(list((goc / 'images/val').glob('*.jpg'))))
print('✅ Trọng số gốc:', (goc / 'litealpr_yolov8n_eff.pt').exists())

In [ ]:
# Bước 3: Đo điểm chuẩn TRƯỚC khi Fine-tune trên tập Val (gồm cả ca đêm lóa pha & chữ OSD đè biển)
from ultralytics import YOLO

TRONG_SO_GOC = '/content/du_lieu_yolo_detect/litealpr_yolov8n_eff.pt'
FILE_YAML = '/content/du_lieu_yolo_detect/data_colab.yaml'

mo_hinh_goc = YOLO(TRONG_SO_GOC)
kq_truoc = mo_hinh_goc.val(data=FILE_YAML, imgsz=640, batch=32, verbose=False)
print(f"📊 TRƯỚC FINE-TUNE -> Precision: {kq_truoc.box.mp:.4f} | Recall: {kq_truoc.box.mr:.4f} | mAP50: {kq_truoc.box.map50:.4f} | mAP50-95: {kq_truoc.box.map:.4f}")

In [ ]:
# Bước 4: Huấn luyện Fine-tune (35 epochs, tối ưu cho biển nhỏ ở xa, đêm tối & bị che khuất một phần)
from ultralytics import YOLO

THU_MUC_LUU_DRIVE = '/content/drive/MyDrive/LITEALPR_FINETUNE_YOLO_DETECT'
mo_hinh = YOLO('/content/du_lieu_yolo_detect/litealpr_yolov8n_eff.pt')

mo_hinh.train(
    data='/content/du_lieu_yolo_detect/data_colab.yaml',
    epochs=35,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.05,
    cos_lr=True,
    warmup_epochs=3.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.6,       # Mở rộng biên độ sáng/tối mạnh cho ảnh ban đêm
    scale=0.6,       # Phóng to/thu nhỏ ±60% để bắt nhạy cả biển ở xa lẫn ở gần
    erasing=0.25,    # Che ngẫu nhiên một phần giúp bắt tốt biển bị chữ OSD/barie che khuất
    degrees=5.0,
    fliplr=0.0,      # Không lật ngang gương để giữ đúng bố cục biển số & chữ OSD
    close_mosaic=8,
    patience=12,
    project=THU_MUC_LUU_DRIVE,
    name='v1_ban_dem_etc',
    exist_ok=True,
)

In [ ]:
# Bước 5: Đánh giá SAU Fine-tune & Xuất file duy nhất yolo_detect_hon_hop.pt ra Google Drive
import shutil
from pathlib import Path
from ultralytics import YOLO

THU_MUC_LUU_DRIVE = Path('/content/drive/MyDrive/LITEALPR_FINETUNE_YOLO_DETECT')
duong_dan_best = THU_MUC_LUU_DRIVE / 'v1_ban_dem_etc' / 'weights' / 'best.pt'
duong_dan_xuat = THU_MUC_LUU_DRIVE / 'yolo_detect_hon_hop.pt'

mo_hinh_moi = YOLO(str(duong_dan_best))
kq_sau = mo_hinh_moi.val(data='/content/du_lieu_yolo_detect/data_colab.yaml', imgsz=640, batch=32, verbose=False)
print(f"🏆 SAU FINE-TUNE   -> Precision: {kq_sau.box.mp:.4f} | Recall: {kq_sau.box.mr:.4f} | mAP50: {kq_sau.box.map50:.4f} | mAP50-95: {kq_sau.box.map:.4f}")

shutil.copy2(duong_dan_best, duong_dan_xuat)
print('✅ ĐÃ LƯU XONG MODEL TẠI GOOGLE DRIVE:', duong_dan_xuat)
!ls -lh "{duong_dan_xuat}"